In [7]:
import pandas as pd
import ast

import csv
from typing import Sequence, Tuple

In [ ]:
##fix_tsv  'L41_408', 'L56_481' <---- malformed ALL_ANSWERS entry

In [2]:
fix_csv = pd.read_csv("../data_raw/full/fixations_A.csv")
ia_csv = pd.read_csv("../data_raw/full/ia_A.csv")


C:\Users\deeth\AppData\Local\Temp\ipykernel_17512\3904489919.py:1: DtypeWarning: Columns (0: CURRENT_FIX_NEAREST_INTEREST_AREA, 1: CURRENT_FIX_NEAREST_INTEREST_AREA_DISTANCE, 2: CURRENT_FIX_PRECISION_MEASURE_RMS_S2S, 3: CURRENT_FIX_PRECISION_MEASURE_SD_WINDOW) have mixed types. Specify dtype option on import or set low_memory=False.
  fix_csv = pd.read_csv("../data_raw/full/fixations_A.csv")


In [8]:
fix_tsv = pd.read_csv(
    "../data_raw/unused/Fixations reports/fixations_A.tsv",
    sep="\t",
    encoding="utf-16",
    engine="python",
    quoting=csv.QUOTE_NONE
)

In [10]:
def find_malformed_recordings(
    df: pd.DataFrame,
    recording_col: str = "RECORDING_SESSION_LABEL",
    trial_col: str = "TRIAL_INDEX",
    answers_col: str = "ALL_ANSWERS",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, list]:
    """
    Identify malformed ALL_ANSWERS entries and the recordings they belong to.

    Returns
    -------
    bad_rows : pd.DataFrame
        Rows where parsing failed.
    bad_recordings_df : pd.DataFrame
        Summary per recording (number of bad rows).
    bad_recordings_list : list
        List of recording IDs to remove.
    """

    def safe_parse_answers(x):
        if pd.isna(x):
            return []
        try:
            return ast.literal_eval(x)
        except Exception:
            return None

    trial_answers = (
        df[[recording_col, trial_col, answers_col]]
        .drop_duplicates(subset=[recording_col, trial_col])
        .sort_values([recording_col, trial_col])
        .copy()
    )

    trial_answers["ALL_ANSWERS_LIST"] = trial_answers[answers_col].apply(safe_parse_answers)

    # row-level issues
    bad_rows = trial_answers[trial_answers["ALL_ANSWERS_LIST"].isna()].copy()

    # recording-level aggregation
    bad_recordings_df = (
        bad_rows.groupby(recording_col)
        .size()
        .reset_index(name="n_bad_rows")
        .sort_values("n_bad_rows", ascending=False)
    )

    bad_recordings_list = bad_recordings_df[recording_col].tolist()

    if verbose:
        print(f"Bad rows: {len(bad_rows)}")
        print(f"Bad recordings: {len(bad_recordings_list)}")

        if len(bad_recordings_list) > 0:
            print("\nTop problematic recordings:")
            print(bad_recordings_df.head(10))

    return bad_rows, bad_recordings_df, bad_recordings_list

In [11]:
def remove_malformed_recordings(
    df: pd.DataFrame,
    bad_recordings: Sequence[str],
    col: str = "RECORDING_SESSION_LABEL",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Remove rows belonging to malformed recordings.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    bad_recordings : list-like
        List of recording IDs to remove.
    col : str
        Column containing recording IDs.
    verbose : bool
        Whether to print summary information.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe.
    """
    before_rows = len(df)
    before_unique = df[col].nunique()

    mask = df[col].isin(bad_recordings)
    removed_rows = mask.sum()
    removed_recordings = df.loc[mask, col].nunique()

    out = df.loc[~mask].copy()

    if verbose:
        print(f"Removed {removed_rows} rows from {removed_recordings} recordings.")
        print(f"Rows: {before_rows} → {len(out)}")
        print(f"Recordings: {before_unique} → {out[col].nunique()}")

    return out

In [12]:
bad_rows, bad_recordings_df, bad_recordings = find_malformed_recordings(fix_tsv)

Bad rows: 19250
Bad recordings: 316

Top problematic recordings:
    RECORDING_SESSION_LABEL  n_bad_rows
30                  l15_214          68
111                 l28_242          68
45                   l17_98          67
103                 l26_376          67
269                 l55_180          67
62                    l1_64          66
98                   l25_56          66
261                 l53_431          66
162                 l36_280          66
114                 l28_380          66


In [46]:
fix_tsv = remove_malformed_recordings(fix_tsv, bad_recordings)

Removed 636393 rows from 314 recordings.
Rows: 713637 → 77244
Recordings: 358 → 44


In [30]:
fix_csv[['CURRENT_FIX_MSG_TEXT_1', 'CURRENT_FIX_MSG_TIME_1']]

,CURRENT_FIX_MSG_TEXT_1,CURRENT_FIX_MSG_TIME_1
0,DISPLAY_ANSWERS,45487
1,.,.
2,.,.
3,.,.
4,.,.
...,...,...
718202,.,.
718203,.,.
718204,.,.
718205,.,.


In [40]:
# one row per participant/trial
trial_answers = (
    fix_tsv[
        ['RECORDING_SESSION_LABEL', 'TRIAL_INDEX', 'ALL_ANSWERS']
    ]
    .drop_duplicates(subset=['RECORDING_SESSION_LABEL', 'TRIAL_INDEX'])
    .sort_values(['RECORDING_SESSION_LABEL', 'TRIAL_INDEX'])
    .copy()
)

# convert string representation of list into actual Python list
trial_answers['ALL_ANSWERS_LIST'] = trial_answers['ALL_ANSWERS'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# previous cumulative list within participant
trial_answers['PREV_ALL_ANSWERS_LIST'] = (
    trial_answers
    .groupby('RECORDING_SESSION_LABEL')['ALL_ANSWERS_LIST']
    .shift(1)
)

# for first trial, keep full list
trial_answers['TRIAL_ANSWERS'] = trial_answers['ALL_ANSWERS_LIST']

# for later trials, keep only the newly added entries
mask = trial_answers['PREV_ALL_ANSWERS_LIST'].notna()

trial_answers.loc[mask, 'TRIAL_ANSWERS'] = trial_answers.loc[mask].apply(
    lambda row: row['ALL_ANSWERS_LIST'][len(row['PREV_ALL_ANSWERS_LIST']):],
    axis=1
)

trial_answers[
    [
        'RECORDING_SESSION_LABEL',
        'TRIAL_INDEX',
        'ALL_ANSWERS',
        'ALL_ANSWERS_LIST',
        'PREV_ALL_ANSWERS_LIST',
        'TRIAL_ANSWERS'
    ]
]

SyntaxError: '(' was never closed (<unknown>, line 1)

In [41]:
import ast
import pandas as pd

def safe_parse_answers(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except Exception:
        return None

trial_answers = (
    fix_tsv[
        ['RECORDING_SESSION_LABEL', 'TRIAL_INDEX', 'ALL_ANSWERS']
    ]
    .drop_duplicates(subset=['RECORDING_SESSION_LABEL', 'TRIAL_INDEX'])
    .sort_values(['RECORDING_SESSION_LABEL', 'TRIAL_INDEX'])
    .copy()
)

trial_answers['ALL_ANSWERS_LIST'] = trial_answers['ALL_ANSWERS'].apply(safe_parse_answers)

bad_rows = trial_answers[trial_answers['ALL_ANSWERS_LIST'].isna()].copy()

print(f"Bad rows: {len(bad_rows)}")
bad_rows[['RECORDING_SESSION_LABEL', 'TRIAL_INDEX', 'ALL_ANSWERS']].head(20)

Bad rows: 19121


,RECORDING_SESSION_LABEL,TRIAL_INDEX,ALL_ANSWERS
664944,l10_202,9,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
664985,l10_202,10,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665020,l10_202,11,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665061,l10_202,12,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665141,l10_202,13,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665174,l10_202,17,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665183,l10_202,18,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665238,l10_202,19,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665267,l10_202,20,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."
665311,l10_202,21,"[('0', 7638), ('0', 13710), ('2', 4109), ('0',..."


In [35]:
fix_tsv[fix_tsv['RECORDING_SESSION_LABEL'] == 'L56_481'][['RECORDING_SESSION_LABEL','TRIAL_INDEX','ALL_ANSWERS']].drop_duplicates()

,RECORDING_SESSION_LABEL,TRIAL_INDEX,ALL_ANSWERS
537279,L56_481,1,"[('0', 30027)]"
537385,L56_481,2,"[('0', 30027), ('3', 25547)]"
537481,L56_481,3,"[('0', 30027), ('3', 25547), ('3', 3105)]"
537499,L56_481,4,"[('0', 30027), ('3', 25547), ('3', 3105), ('1'..."
537626,L56_481,5,"[('0', 30027), ('3', 25547), ('3', 3105), ('1'..."
...,...,...,...
539999,L56_481,66,"[('0', 30027), ('3', 25547), ('3', 3105), ('1'..."
540024,L56_481,67,"[('0', 30027), ('3', 25547), ('3', 3105), ('1'..."
540054,L56_481,68,"[('0', 30027), ('3', 25547), ('3', 3105), ('1'..."
540122,L56_481,69,"[('0', 30027), ('3', 25547), ('3', 3105), ('1'..."


In [22]:
fix_csv[fix_csv['participant_id'] == 'L41_408'][['CURRENT_FIX_MSG_TEXT_1', 'CURRENT_FIX_MSG_TIME_1']]

,CURRENT_FIX_MSG_TEXT_1,CURRENT_FIX_MSG_TIME_1
551560,DISPLAY_ANSWERS,32877
551561,.,.
551562,.,.
551563,.,.
551564,.,.
...,...,...
553097,.,.
553098,.,.
553099,.,.
553100,.,.


In [21]:
ia_csv[ia_csv['participant_id'] == 'L41_408']

KeyError: "None of [Index(['CURRENT_FIX_MSG_TEXT_1', 'CURRENT_FIX_MSG_TIME_1'], dtype='str')] are in the [columns]"